# Phase 10 Model Optimization

Authorized Standard-budget execution for the Reto Tokio / GCI World NFL Draft Prediction project.

This notebook executes bounded HPO only for the authorized candidates: M1 LogisticRegression and conditional CatBoost. It does not create submissions, use leaderboard feedback, open Phase 11, declare a final winner, or write to `logs/experiment_log.csv`.

## 1. Setup And Scope

**Objective.** Configure Phase 10 execution, capture environment and Git context, and enforce non-actions.
**Inputs.** Authorization note, repository state, project venv.
**Method.** Define constants, paths, write guards, metric helpers, and command helpers.
**Expected output.** A reproducible execution context and guarded artifact paths.
**Risk controlled.** Wrong commit, accidental overwrite, main-log mutation, scope creep.

In [1]:
# 1. Setup and scope
from __future__ import annotations

import csv
import hashlib
import itertools
import json
import math
import os
import subprocess
import sys
import textwrap
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_SEED = 42
EXPERIMENT_ID = "phase10_model_optimization_v1"
RUN_ID = "phase10_standard_20260619_0152"
AUTHORIZED_HEAD = "fc7a625dfd53b08b5e53ee9f1aeae9b47a2ec6a8"
AUTHORIZED_PLANNING_HEAD = "e894dd2dc15589dfe3a281129eacbfde95e00ccf"
FOLD_SHA16_EXPECTED = "96937649526bcadb"
REPO_ROOT = Path(r"C:\GitHub\reto_Tokio")
CATBOOST_PYTHON = Path(r"C:\tmp\reto_tokio_phase8_wave2_env\Scripts\python.exe")
CATBOOST_TMP_JSON = Path(r"C:\tmp") / f"{RUN_ID}_catboost_result.json"
CATBOOST_HELPER = Path(r"C:\tmp") / f"{RUN_ID}_catboost_helper.py"

assert (REPO_ROOT / ".git").exists(), f"Repo root not found: {REPO_ROOT}"
os.chdir(REPO_ROOT)

def run_cmd(args: list[str]) -> str:
    result = subprocess.run(args, cwd=REPO_ROOT, text=True, capture_output=True, check=True)
    return result.stdout.strip()

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_write_csv(df: pd.DataFrame, path: Path) -> None:
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

def safe_write_text(text: str, path: Path) -> None:
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing artifact: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")

def csv_rows(path: Path) -> int | str:
    try:
        return len(pd.read_csv(path))
    except Exception:
        return ""

def markdown_table(rows: list[dict], columns: list[str], max_rows: int | None = None) -> str:
    if max_rows is not None:
        rows = rows[:max_rows]
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        vals = []
        for col in columns:
            val = row.get(col, "")
            if isinstance(val, (float, np.floating)):
                vals.append(f"{float(val):.10f}")
            else:
                vals.append(str(val).replace("|", "\\|"))
        body.append("| " + " | ".join(vals) + " |")
    return "\n".join([header, sep] + body)

def positive_class_proba(estimator, X):
    classes = list(estimator.classes_)
    if classes.count(1) != 1:
        raise RuntimeError(f"Estimator classes_ does not contain label 1 exactly once: {classes}")
    idx = classes.index(1)
    proba = estimator.predict_proba(X)[:, idx]
    if not np.isfinite(proba).all() or np.any((proba < 0) | (proba > 1)):
        raise RuntimeError("Invalid predicted probabilities")
    return proba

def auc_or_nan(y_true, y_score) -> float:
    if pd.Series(y_true).nunique() < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))

def ap_or_nan(y_true, y_score) -> float:
    if pd.Series(y_true).nunique() < 2:
        return float("nan")
    return float(average_precision_score(y_true, y_score))

paths = {
    "notebook": REPO_ROOT / "notebooks" / "10_phase10_model_optimization.ipynb",
    "m1_oof": REPO_ROOT / "outputs" / "oof" / f"phase10_model_optimization_{RUN_ID}_m1_logistic_regression_tuned_oof_predictions.csv",
    "catboost_oof": REPO_ROOT / "outputs" / "oof" / f"phase10_model_optimization_{RUN_ID}_catboost_tuned_oof_predictions.csv",
    "hpo_results": REPO_ROOT / "outputs" / "validation" / f"phase10_model_optimization_{RUN_ID}_hpo_results.csv",
    "model_summary": REPO_ROOT / "outputs" / "validation" / f"phase10_model_optimization_{RUN_ID}_model_summary.csv",
    "fold_metrics": REPO_ROOT / "outputs" / "validation" / f"phase10_model_optimization_{RUN_ID}_fold_metrics.csv",
    "slice_report": REPO_ROOT / "outputs" / "validation" / f"phase10_model_optimization_{RUN_ID}_slice_report.csv",
    "topk_quantile": REPO_ROOT / "outputs" / "validation" / f"phase10_model_optimization_{RUN_ID}_topk_quantile.csv",
    "score_distribution": REPO_ROOT / "outputs" / "validation" / f"phase10_model_optimization_{RUN_ID}_score_distribution.csv",
    "selection_bias_report": REPO_ROOT / "outputs" / "reports" / f"phase10_model_optimization_{RUN_ID}_selection_bias_warning_report.md",
    "validation_report": REPO_ROOT / "outputs" / "reports" / f"phase10_model_optimization_{RUN_ID}_validation_report.md",
    "candidate_log": REPO_ROOT / "outputs" / "reports" / f"phase10_model_optimization_{RUN_ID}_experiment_log_candidate.csv",
    "artifact_manifest": REPO_ROOT / "outputs" / "reports" / f"phase10_model_optimization_{RUN_ID}_artifact_manifest.csv",
}

for key, path in paths.items():
    if key == "notebook":
        continue
    if path.exists():
        raise FileExistsError(f"Phase 10 artifact collision: {path}")

head = run_cmd(["git", "rev-parse", "HEAD"])
short_head = run_cmd(["git", "rev-parse", "--short", "HEAD"])
if head != AUTHORIZED_HEAD:
    raise RuntimeError(f"HEAD mismatch: expected {AUTHORIZED_HEAD}, observed {head}")
if run_cmd(["git", "diff", "--cached", "--name-only"]):
    raise RuntimeError("Staged files exist before execution")
forbidden = run_cmd(["git", "diff", "--name-only", "--", "data/input", "notebooks/_official", "references", "outputs/submissions", "logs/experiment_log.csv", ".vscode/settings.json"])
if forbidden:
    raise RuntimeError(f"Forbidden tracked diffs before execution: {forbidden}")
diff_check = subprocess.run(["git", "diff", "--check"], cwd=REPO_ROOT, text=True, capture_output=True)
if diff_check.returncode != 0:
    raise RuntimeError(diff_check.stdout + diff_check.stderr)

authorization_path = REPO_ROOT / "docs" / "10_model_optimization" / "phase10_project_authorization_note.md"
required_docs = [
    authorization_path,
    REPO_ROOT / "docs" / "10_model_optimization" / "phase10_master_planning_brief.md",
    REPO_ROOT / "docs" / "10_model_optimization" / "phase10_operator_runbook.md",
    REPO_ROOT / "docs" / "10_model_optimization" / "prompt_codex_phase10_execution_plan.md",
    REPO_ROOT / "docs" / "09_auc_ranking_diagnostics" / "phase9b_lite_transition_memo.md",
    REPO_ROOT / "docs" / "09_auc_ranking_diagnostics" / "phase9a_acceptance.md",
    REPO_ROOT / "docs" / "09_auc_ranking_diagnostics" / "phase9a_improvement_backlog.md",
    REPO_ROOT / "docs" / "05_methodology" / "validation_protocol_phase6.md",
    REPO_ROOT / "docs" / "05_methodology" / "leakage_checklist_phase6.md",
]
missing_docs = [str(p) for p in required_docs if not p.exists()]
if missing_docs:
    raise FileNotFoundError("Required docs missing: " + "; ".join(missing_docs))

auth_text = authorization_path.read_text(encoding="utf-8")
for required_phrase in [
    "Budget mode: Standard",
    "Gate 5 is explicitly waived",
    "M1 LogisticRegression",
    "CatBoost",
    "Phase 11 remains locked",
]:
    if required_phrase not in auth_text:
        raise RuntimeError(f"Authorization note missing phrase: {required_phrase}")

main_log_path = REPO_ROOT / "logs" / "experiment_log.csv"
main_log_sha_before = sha256_file(main_log_path)

env_summary = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
}

print(f"Phase 10 setup complete at HEAD {short_head}")
print(f"Run ID: {RUN_ID}")

Phase 10 setup complete at HEAD fc7a625
Run ID: phase10_standard_20260619_0152


### Interpretation

- **Main result:** Execution is authorized from the expected commit and protected paths are clean.
- **Methodological reading:** Phase 10 can proceed only as a bounded HPO run on F2 and frozen folds.
- **Risk or warning:** Optuna availability is checked later; if absent, bounded config search remains within the authorized `trials/configs` language but is recorded as a warning.
- **Diagnostic decision:** Continue to fold, feature, and baseline integrity checks.

## 2. Data, Folds, Features And Baselines

**Objective.** Load official train/test only for permitted checks, reconstruct F2 row-wise features, and load persisted OOF baselines.
**Inputs.** `data/input/train.csv`, `data/input/test.csv`, frozen fold file, M0/M1/CatBoost OOF.
**Method.** Verify fold SHA, row counts, F2 columns, School exclusion, and baseline AUC reproduction.
**Expected output.** Valid F2 matrix and baseline OOF dictionaries.
**Risk controlled.** Leakage, row misalignment, wrong target, and anchor drift.

In [2]:
# 2. Data, folds, features and baselines
train = pd.read_csv(REPO_ROOT / "data" / "input" / "train.csv")
test = pd.read_csv(REPO_ROOT / "data" / "input" / "test.csv")
fold_path = REPO_ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
folds = pd.read_csv(fold_path)

if sha256_file(fold_path)[:16] != FOLD_SHA16_EXPECTED:
    raise RuntimeError("Frozen fold SHA mismatch")
if len(train) != 2781 or len(folds) != 2781 or set(folds["fold"].unique()) != set(range(5)):
    raise RuntimeError("Train/fold integrity check failed")
if list(train["Id"]) != list(folds["Id"]):
    raise RuntimeError("Fold Id order does not align with train")
if "Drafted" in test.columns:
    raise RuntimeError("Test data contains target column")

base_features = [
    "Year", "Age", "Height", "Weight", "Sprint_40yd", "Vertical_Jump",
    "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle",
    "Player_Type", "Position_Type", "Position",
]
missing_flag_columns = [
    "Age_missing", "Sprint_40yd_missing", "Vertical_Jump_missing",
    "Bench_Press_Reps_missing", "Broad_Jump_missing", "Agility_3cone_missing",
    "Shuttle_missing",
]
physical_test_columns = [
    "Sprint_40yd", "Vertical_Jump", "Bench_Press_Reps",
    "Broad_Jump", "Agility_3cone", "Shuttle",
]
numeric_features = [
    "Year", "Age", "Height", "Weight", "Sprint_40yd", "Vertical_Jump",
    "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle",
    *missing_flag_columns, "available_measurement_count",
]
categorical_features = ["Player_Type", "Position_Type", "Position"]
f2_features = numeric_features + categorical_features

def build_f2(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in base_features + ["Id", "Drafted"] if c not in df.columns and c != "Drafted"]
    if missing:
        raise RuntimeError(f"Missing expected official columns: {missing}")
    out = df[base_features].copy()
    out["Age_missing"] = df["Age"].isna().astype(int)
    for col in physical_test_columns:
        out[f"{col}_missing"] = df[col].isna().astype(int)
    out["available_measurement_count"] = df[physical_test_columns].notna().sum(axis=1).astype(int)
    forbidden = {"Id", "Drafted", "School"}
    if forbidden.intersection(out.columns):
        raise RuntimeError(f"Forbidden column entered F2 matrix: {forbidden.intersection(out.columns)}")
    return out[f2_features]

X = build_f2(train)
y = train["Drafted"].astype(int)
fold_ids = folds["fold"].astype(int).to_numpy()
if set(y.unique()) != {0, 1}:
    raise RuntimeError("Target is not binary 0/1")

preprocessor_m1 = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ],
    remainder="drop",
)
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_features),
    ],
    remainder="drop",
)

baseline_files = {
    "m0_random_forest_frozen": REPO_ROOT / "outputs" / "oof" / "phase8_model_family_comparison_v1_m0_random_forest_frozen_oof_predictions.csv",
    "m1_logistic_regression_baseline": REPO_ROOT / "outputs" / "oof" / "phase8_model_family_comparison_v1_m1_logistic_regression_oof_predictions.csv",
    "catboost_baseline": REPO_ROOT / "outputs" / "oof" / "phase8_wave2_external_gbdt_v1_catboost_oof_predictions.csv",
}
expected_baseline_auc = {
    "m0_random_forest_frozen": 0.8116502602456482,
    "m1_logistic_regression_baseline": 0.8270821069632867,
    "catboost_baseline": 0.8202943968641223,
}
baseline_oofs = {}
for key, path in baseline_files.items():
    df = pd.read_csv(path)
    expected_cols = ["Id", "fold", "y_true", "y_pred_proba"]
    if list(df.columns) != expected_cols or len(df) != 2781:
        raise RuntimeError(f"Baseline OOF failed schema/row check: {key}")
    if not df["y_pred_proba"].between(0, 1).all() or not np.isfinite(df["y_pred_proba"]).all():
        raise RuntimeError(f"Baseline OOF has invalid proba: {key}")
    if list(df["Id"]) != list(train["Id"]) or list(df["fold"]) != list(folds["fold"]) or list(df["y_true"].astype(int)) != list(y):
        raise RuntimeError(f"Baseline OOF alignment failed: {key}")
    auc = float(roc_auc_score(df["y_true"], df["y_pred_proba"]))
    if abs(auc - expected_baseline_auc[key]) > 1e-9:
        raise RuntimeError(f"Baseline AUC drift for {key}: {auc}")
    baseline_oofs[key] = df

print("F2, folds, and baselines verified")
print({k: roc_auc_score(v["y_true"], v["y_pred_proba"]) for k, v in baseline_oofs.items()})

F2, folds, and baselines verified
{'m0_random_forest_frozen': 0.8116502602456482, 'm1_logistic_regression_baseline': 0.8270821069632867, 'catboost_baseline': 0.8202943968641223}


### Interpretation

- **Main result:** F2, frozen folds, and persisted M0/M1/CatBoost baselines are aligned.
- **Methodological reading:** All Phase 10 comparisons can be paired on the same row/fold/target substrate.
- **Risk or warning:** `School` remains diagnostic-only and is excluded from F2.
- **Diagnostic decision:** Continue to B5/B3 pre-HPO diagnostics.

## 3. Pre-HPO Diagnostics

**Objective.** Perform required B5 CatBoost slice-instability diagnosis and B3 M1 stability framing before tuning.
**Inputs.** Existing baseline OOF files and official train rows for diagnostic slices.
**Method.** Compute known warning slices and determine whether CatBoost limited HPO can proceed as secondary/observe.
**Expected output.** A diagnosis table and proceed/stop decision for CatBoost HPO.
**Risk controlled.** Tuning CatBoost before acknowledging robust-slice instability.

In [3]:
# 3. Pre-HPO diagnostics
diag = train[["Id", "Year", "Age", "Player_Type", "Position_Type", "Position", "School", *physical_test_columns]].copy()
diag["Age_missing"] = diag["Age"].isna().astype(int).astype(str)
diag["available_measurement_count"] = diag[physical_test_columns].notna().sum(axis=1).astype(int)
diag["available_measurement_count_str"] = diag["available_measurement_count"].astype(str)
diag["measurement_completeness_group"] = pd.cut(
    diag["available_measurement_count"],
    bins=[-1, 0, 3, 5, 6],
    labels=["none", "low", "partial", "complete"],
).astype(str)
school_counts = diag["School"].value_counts(dropna=False)
diag["frequent_vs_rare_school_group"] = diag["School"].map(school_counts).fillna(0).apply(lambda c: "frequent" if c >= 5 else "rare")
diag["Year"] = diag["Year"].astype("Int64").astype(str)
for c in ["Player_Type", "Position_Type", "Position"]:
    diag[c] = diag[c].fillna("missing").astype(str)

slice_specs = [
    ("Player_Type", "Player_Type", 50),
    ("Position_Type", "Position_Type", 50),
    ("Year", "Year", 50),
    ("Age_missing", "Age_missing", 50),
    ("available_measurement_count", "available_measurement_count_str", 50),
    ("measurement_completeness_group", "measurement_completeness_group", 50),
    ("frequent_vs_rare_school_group", "frequent_vs_rare_school_group", 50),
    ("Position", "Position", 50),
]

def compute_slice_auc(oof_df: pd.DataFrame, ids: set[int]) -> tuple[float, int, int, int, str]:
    sub = oof_df[oof_df["Id"].isin(ids)]
    n = int(len(sub))
    n_pos = int((sub["y_true"] == 1).sum())
    n_neg = int((sub["y_true"] == 0).sum())
    if n < 50:
        return np.nan, n, n_pos, n_neg, "skipped_too_small"
    if n_pos == 0 or n_neg == 0:
        return np.nan, n, n_pos, n_neg, "skipped_one_class"
    return float(roc_auc_score(sub["y_true"], sub["y_pred_proba"])), n, n_pos, n_neg, "computed"

b5_records = []
for public_name, col, min_n in slice_specs:
    for value in sorted(diag[col].dropna().unique(), key=lambda x: str(x)):
        ids = set(diag.loc[diag[col] == value, "Id"].astype(int))
        m0_auc, n, n_pos, n_neg, status = compute_slice_auc(baseline_oofs["m0_random_forest_frozen"], ids)
        cat_auc, _, _, _, cat_status = compute_slice_auc(baseline_oofs["catboost_baseline"], ids)
        m1_auc, _, _, _, _ = compute_slice_auc(baseline_oofs["m1_logistic_regression_baseline"], ids)
        delta_cat_m0 = cat_auc - m0_auc if status == "computed" and cat_status == "computed" else np.nan
        b5_records.append({
            "slice_name": public_name,
            "slice_value": str(value),
            "n": n,
            "n_positive": n_pos,
            "n_negative": n_neg,
            "m0_auc": m0_auc,
            "m1_auc": m1_auc,
            "catboost_auc": cat_auc,
            "catboost_delta_vs_m0": delta_cat_m0,
            "status": status,
            "catboost_drop_gt_0_02_vs_m0": bool(status == "computed" and delta_cat_m0 < -0.02),
            "fragile_positive_support_lt_20": bool(status == "computed" and n_pos < 20),
        })
b5_df = pd.DataFrame(b5_records)
catboost_robust_warnings = b5_df[(b5_df["catboost_drop_gt_0_02_vs_m0"]) & (~b5_df["fragile_positive_support_lt_20"])]
catboost_hpo_proceed = True
catboost_hpo_reason = (
    "B5 diagnosis completed; CatBoost instability is confirmed and will be treated as a guardrail. "
    "No leakage or data-integrity blocker was found, so limited secondary HPO may proceed under the authorization note."
)

m1_baseline_fold_aucs = []
for fold in range(5):
    fold_df = baseline_oofs["m1_logistic_regression_baseline"][baseline_oofs["m1_logistic_regression_baseline"]["fold"] == fold]
    m1_baseline_fold_aucs.append(float(roc_auc_score(fold_df["y_true"], fold_df["y_pred_proba"])))

print("B5 CatBoost robust warning count:", len(catboost_robust_warnings))
print("CatBoost HPO proceed:", catboost_hpo_proceed)

B5 CatBoost robust warning count: 7
CatBoost HPO proceed: True


### Interpretation

- **Main result:** B5 diagnosis is completed before CatBoost HPO.
- **Methodological reading:** CatBoost instability is a guardrail, not a leakage finding.
- **Risk or warning:** CatBoost may not be promoted cleanly if robust-slice warnings persist after tuning.
- **Diagnostic decision:** Proceed with limited CatBoost HPO as secondary/observe because the authorization explicitly allows it after B5 and no integrity blocker was found.

## 4. M1 LogisticRegression Bounded HPO

**Objective.** Run the authorized M1 bounded HPO under the Standard budget.
**Inputs.** F2 features, frozen folds, pre-registered LogisticRegression search space.
**Method.** Sequential deterministic config search over 50 configurations; fold-safe preprocessing in every trial.
**Expected output.** M1 HPO trial table and best tuned OOF vector.
**Risk controlled.** Preprocessing leakage, search-space drift, excessive HPO budget, wrong probability class.

In [4]:
# 4. M1 LogisticRegression bounded HPO
try:
    import optuna  # noqa: F401
    optuna_available = True
except Exception:
    optuna_available = False

m1_configs = []
for i, c_val in enumerate(np.logspace(-3, 2, 50)):
    penalty = "l2" if i % 2 == 0 else "l1"
    class_weight = None if (i // 2) % 2 == 0 else "balanced"
    solver = "lbfgs" if penalty == "l2" else "saga"
    m1_configs.append({
        "trial_index": i,
        "candidate_family": "m1_logistic_regression",
        "C": float(c_val),
        "penalty": penalty,
        "class_weight": class_weight,
        "solver": solver,
        "max_iter": 3000,
    })

def evaluate_m1_config(config: dict) -> tuple[dict, pd.DataFrame]:
    oof = np.full(len(X), np.nan, dtype=float)
    fold_scores = []
    for fold in range(5):
        train_idx = np.where(fold_ids != fold)[0]
        valid_idx = np.where(fold_ids == fold)[0]
        lr = LogisticRegression(
            C=config["C"],
            penalty=config["penalty"],
            class_weight=config["class_weight"],
            solver=config["solver"],
            max_iter=config["max_iter"],
            random_state=PROJECT_SEED,
            n_jobs=None,
        )
        pipe = Pipeline([
            ("preprocess", preprocessor_m1),
            ("model", lr),
        ])
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = positive_class_proba(pipe.named_steps["model"], pipe.named_steps["preprocess"].transform(X.iloc[valid_idx]))
        oof[valid_idx] = proba
        fold_scores.append(float(roc_auc_score(y.iloc[valid_idx], proba)))
    if np.isnan(oof).any():
        raise RuntimeError("M1 OOF contains missing predictions")
    oof_df = pd.DataFrame({
        "Id": train["Id"].astype(int),
        "fold": fold_ids.astype(int),
        "y_true": y.astype(int),
        "y_pred_proba": oof,
    })
    result = {
        **config,
        "hpo_strategy": "deterministic_config_search_no_optuna_available" if not optuna_available else "deterministic_config_search_optuna_available_not_used",
        "oof_auc": float(roc_auc_score(y, oof)),
        "fold_mean_auc": float(np.mean(fold_scores)),
        "fold_std_auc": float(np.std(fold_scores, ddof=0)),
        "fold_auc_scores": "|".join(f"{v:.12f}" for v in fold_scores),
        "warning_count": 0,
    }
    return result, oof_df

m1_hpo_rows = []
m1_oof_by_trial = {}
for config in m1_configs:
    result, oof_df = evaluate_m1_config(config)
    m1_hpo_rows.append(result)
    m1_oof_by_trial[result["trial_index"]] = oof_df

m1_hpo_df = pd.DataFrame(m1_hpo_rows).sort_values(["oof_auc", "fold_mean_auc"], ascending=False).reset_index(drop=True)
m1_best_trial_index = int(m1_hpo_df.loc[0, "trial_index"])
m1_best_oof = m1_oof_by_trial[m1_best_trial_index]
m1_best_config = m1_hpo_df.loc[0].to_dict()
print("Optuna available:", optuna_available)
print("M1 trials/configs:", len(m1_hpo_df))
print("M1 best trial:", m1_best_trial_index, "AUC:", m1_best_config["oof_auc"])

Optuna available: False
M1 trials/configs: 50
M1 best trial: 31 AUC: 0.8274819177762125


### Interpretation

- **Main result:** M1 HPO uses exactly 50 bounded configurations.
- **Methodological reading:** The search stays inside the pre-registered space and uses frozen folds.
- **Risk or warning:** Optuna was unavailable in `.venv`; deterministic config search is recorded as a warning while preserving the authorized `trials/configs` cap and search space.
- **Diagnostic decision:** Continue to CatBoost limited HPO through the separate Wave 2 environment.

## 5. CatBoost Limited HPO

**Objective.** Run conditional limited CatBoost HPO after B5 diagnosis, using the separate Wave 2 environment.
**Inputs.** F2 features, frozen folds, pre-registered CatBoost search space, existing Wave 2 env.
**Method.** Execute a helper in the separate CatBoost environment with `cat_features=[]` and fold-safe preprocessing.
**Expected output.** CatBoost HPO trial table and tuned OOF vector.
**Risk controlled.** `.venv` contamination, School leakage, native categorical leakage, HPO budget drift.

In [5]:
# 5. CatBoost limited HPO in separate env
if not catboost_hpo_proceed:
    raise RuntimeError("CatBoost HPO blocked by B5 diagnosis")
if not CATBOOST_PYTHON.exists():
    raise FileNotFoundError(f"Separate CatBoost env missing: {CATBOOST_PYTHON}")

catboost_helper_code = r"""
from __future__ import annotations
import itertools
import json
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

REPO_ROOT = Path(r'C:\GitHub\reto_Tokio')
OUTPUT_JSON = Path(r'__OUTPUT_JSON__')
PROJECT_SEED = 42
physical_test_columns = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
numeric_features = [
    'Year', 'Age', 'Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
    'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle',
    'Age_missing', 'Sprint_40yd_missing', 'Vertical_Jump_missing',
    'Bench_Press_Reps_missing', 'Broad_Jump_missing', 'Agility_3cone_missing',
    'Shuttle_missing', 'available_measurement_count',
]
categorical_features = ['Player_Type', 'Position_Type', 'Position']
base_features = [
    'Year', 'Age', 'Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump',
    'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle',
    'Player_Type', 'Position_Type', 'Position',
]
def build_f2(df):
    out = df[base_features].copy()
    out['Age_missing'] = df['Age'].isna().astype(int)
    for col in physical_test_columns:
        out[f'{col}_missing'] = df[col].isna().astype(int)
    out['available_measurement_count'] = df[physical_test_columns].notna().sum(axis=1).astype(int)
    forbidden = {'Id', 'Drafted', 'School'}
    if forbidden.intersection(out.columns):
        raise RuntimeError(f'Forbidden column entered CatBoost F2 matrix: {forbidden.intersection(out.columns)}')
    return out[numeric_features + categorical_features]
def positive_class_proba(model, X):
    classes = list(model.classes_)
    if classes.count(1) != 1:
        raise RuntimeError(f'CatBoost classes_ invalid: {classes}')
    idx = classes.index(1)
    proba = model.predict_proba(X)[:, idx]
    if not np.isfinite(proba).all() or np.any((proba < 0) | (proba > 1)):
        raise RuntimeError('Invalid CatBoost probabilities')
    return proba
train = pd.read_csv(REPO_ROOT / 'data' / 'input' / 'train.csv')
folds = pd.read_csv(REPO_ROOT / 'outputs' / 'folds' / 'phase6_rf_sanity_baseline_v1_fold_assignments.csv')
X = build_f2(train)
y = train['Drafted'].astype(int)
fold_ids = folds['fold'].astype(int).to_numpy()
preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_features),
    ],
    remainder='drop',
)
config_rows = []
learning_rates = np.geomspace(0.01, 0.2, 10)
for i in range(30):
    config_rows.append({
        'trial_index': i,
        'candidate_family': 'catboost',
        'depth': [4, 6, 8][i % 3],
        'learning_rate': float(learning_rates[i % len(learning_rates)]),
        'l2_leaf_reg': [1, 3, 5, 9][(i // 3) % 4],
        'iterations': [200, 400, 800][(i // 5) % 3],
        'border_count': [64, 128][(i // 7) % 2],
        'random_seed': PROJECT_SEED,
    })
hpo_rows = []
oof_by_trial = {}
for config in config_rows:
    oof = np.full(len(X), np.nan, dtype=float)
    fold_scores = []
    for fold in range(5):
        train_idx = np.where(fold_ids != fold)[0]
        valid_idx = np.where(fold_ids == fold)[0]
        pre = preprocessor_tree
        X_train = pre.fit_transform(X.iloc[train_idx], y.iloc[train_idx])
        X_valid = pre.transform(X.iloc[valid_idx])
        model = CatBoostClassifier(
            loss_function='Logloss',
            eval_metric='AUC',
            depth=config['depth'],
            learning_rate=config['learning_rate'],
            l2_leaf_reg=config['l2_leaf_reg'],
            iterations=config['iterations'],
            border_count=config['border_count'],
            random_seed=PROJECT_SEED,
            allow_writing_files=False,
            verbose=False,
            thread_count=-1,
        )
        model.fit(X_train, y.iloc[train_idx], cat_features=[])
        proba = positive_class_proba(model, X_valid)
        oof[valid_idx] = proba
        fold_scores.append(float(roc_auc_score(y.iloc[valid_idx], proba)))
    if np.isnan(oof).any():
        raise RuntimeError('CatBoost OOF contains missing predictions')
    result = {
        **config,
        'hpo_strategy': 'deterministic_config_search_separate_catboost_env',
        'oof_auc': float(roc_auc_score(y, oof)),
        'fold_mean_auc': float(np.mean(fold_scores)),
        'fold_std_auc': float(np.std(fold_scores, ddof=0)),
        'fold_auc_scores': '|'.join(f'{v:.12f}' for v in fold_scores),
    }
    hpo_rows.append(result)
    oof_by_trial[str(config['trial_index'])] = oof.tolist()
best = sorted(hpo_rows, key=lambda r: (r['oof_auc'], r['fold_mean_auc']), reverse=True)[0]
payload = {
    'hpo_rows': hpo_rows,
    'best_trial_index': best['trial_index'],
    'best_oof': oof_by_trial[str(best['trial_index'])],
    'versions': {},
}
try:
    import catboost, sklearn, pandas, numpy
    payload['versions'] = {
        'catboost': catboost.__version__,
        'sklearn': sklearn.__version__,
        'pandas': pandas.__version__,
        'numpy': numpy.__version__,
    }
except Exception as exc:
    payload['versions_error'] = str(exc)
OUTPUT_JSON.write_text(json.dumps(payload), encoding='utf-8')
"""
catboost_helper_code = catboost_helper_code.replace("__OUTPUT_JSON__", str(CATBOOST_TMP_JSON).replace("\\", "\\\\"))
CATBOOST_HELPER.write_text(catboost_helper_code, encoding="utf-8", newline="\n")

env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
subprocess.run([str(CATBOOST_PYTHON), str(CATBOOST_HELPER)], cwd=REPO_ROOT, check=True, env=env)
cat_payload = json.loads(CATBOOST_TMP_JSON.read_text(encoding="utf-8"))
cat_hpo_df = pd.DataFrame(cat_payload["hpo_rows"]).sort_values(["oof_auc", "fold_mean_auc"], ascending=False).reset_index(drop=True)
cat_best_trial_index = int(cat_hpo_df.loc[0, "trial_index"])
cat_best_oof = pd.DataFrame({
    "Id": train["Id"].astype(int),
    "fold": fold_ids.astype(int),
    "y_true": y.astype(int),
    "y_pred_proba": np.array(cat_payload["best_oof"], dtype=float),
})
cat_versions = cat_payload.get("versions", {})
print("CatBoost trials/configs:", len(cat_hpo_df))
print("CatBoost best trial:", cat_best_trial_index, "AUC:", float(cat_hpo_df.loc[0, "oof_auc"]))
print("CatBoost env versions:", cat_versions)

CatBoost trials/configs: 30
CatBoost best trial: 10 AUC: 0.830320858101755
CatBoost env versions: {'catboost': '1.2.10', 'sklearn': '1.9.0', 'pandas': '3.0.3', 'numpy': '2.4.6'}


### Interpretation

- **Main result:** CatBoost limited HPO uses 30 bounded configurations in the separate Wave 2 environment.
- **Methodological reading:** CatBoost remains secondary/observe; tuning does not erase inherited slice instability.
- **Risk or warning:** CatBoost uses `cat_features=[]` and the same fold-safe pre-encoded F2 design; School remains excluded.
- **Diagnostic decision:** Continue to metrics, slice checks, and artifact writing.

## 6. Metrics, Diagnostics And Artifact Writing

**Objective.** Compute global/fold/slice/top-k/score-distribution diagnostics and write all authorized artifacts.
**Inputs.** Baseline OOF, tuned M1 OOF, tuned CatBoost OOF, HPO trial tables.
**Method.** Pair all models on frozen folds, generate reports, candidate log, and manifest.
**Expected output.** Versioned Phase 10 artifacts under `outputs/oof`, `outputs/validation`, and `outputs/reports`.
**Risk controlled.** Selection bias, hidden slice regression, log mutation, and artifact drift.

In [6]:
# 6. Metrics, diagnostics and artifact writing
def validate_oof(name: str, df: pd.DataFrame) -> None:
    expected_cols = ["Id", "fold", "y_true", "y_pred_proba"]
    if list(df.columns) != expected_cols:
        raise RuntimeError(f"{name} OOF schema mismatch")
    if len(df) != 2781:
        raise RuntimeError(f"{name} OOF row count mismatch")
    if list(df["Id"]) != list(train["Id"]) or list(df["fold"]) != list(folds["fold"]) or list(df["y_true"].astype(int)) != list(y):
        raise RuntimeError(f"{name} OOF alignment mismatch")
    if not np.isfinite(df["y_pred_proba"]).all() or not df["y_pred_proba"].between(0, 1).all():
        raise RuntimeError(f"{name} OOF invalid probabilities")

validate_oof("m1_tuned", m1_best_oof)
validate_oof("catboost_tuned", cat_best_oof)

all_oofs = {
    "m0_random_forest_frozen": baseline_oofs["m0_random_forest_frozen"],
    "m1_logistic_regression_baseline": baseline_oofs["m1_logistic_regression_baseline"],
    "catboost_baseline": baseline_oofs["catboost_baseline"],
    "m1_logistic_regression_tuned": m1_best_oof,
    "catboost_tuned": cat_best_oof,
}

fold_auc_lookup = {}
summary_rows = []
fold_rows = []
for model_key, df in all_oofs.items():
    fold_scores = []
    for fold in range(5):
        fold_df = df[df["fold"] == fold]
        auc = float(roc_auc_score(fold_df["y_true"], fold_df["y_pred_proba"]))
        ap = float(average_precision_score(fold_df["y_true"], fold_df["y_pred_proba"]))
        fold_auc_lookup[(model_key, fold)] = auc
        fold_scores.append(auc)
        fold_rows.append({
            "experiment_id": EXPERIMENT_ID,
            "run_id": RUN_ID,
            "model_key": model_key,
            "fold": fold,
            "n": int(len(fold_df)),
            "n_positive": int((fold_df["y_true"] == 1).sum()),
            "n_negative": int((fold_df["y_true"] == 0).sum()),
            "roc_auc": auc,
            "average_precision": ap,
            "delta_auc_vs_m0": np.nan,
            "delta_auc_vs_m1_baseline": np.nan,
        })
    auc = float(roc_auc_score(df["y_true"], df["y_pred_proba"]))
    summary_rows.append({
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "model_key": model_key,
        "run_status": "tuned" if model_key.endswith("_tuned") else "baseline_loaded",
        "oof_auc": auc,
        "average_precision": float(average_precision_score(df["y_true"], df["y_pred_proba"])),
        "negative_average_precision": float(average_precision_score(1 - df["y_true"], 1 - df["y_pred_proba"])),
        "brier_score": float(brier_score_loss(df["y_true"], df["y_pred_proba"])),
        "delta_vs_m0": auc - float(roc_auc_score(all_oofs["m0_random_forest_frozen"]["y_true"], all_oofs["m0_random_forest_frozen"]["y_pred_proba"])),
        "delta_vs_m1": auc - float(roc_auc_score(all_oofs["m1_logistic_regression_baseline"]["y_true"], all_oofs["m1_logistic_regression_baseline"]["y_pred_proba"])),
        "fold_mean_auc": float(np.mean(fold_scores)),
        "fold_std_auc": float(np.std(fold_scores, ddof=0)),
        "fold_auc_scores": "|".join(f"{v:.12f}" for v in fold_scores),
        "same_sign_positive_folds_vs_m0": np.nan,
        "same_sign_positive_folds_vs_m1": np.nan,
    })
summary_df = pd.DataFrame(summary_rows)
for idx, row in summary_df.iterrows():
    model_key = row["model_key"]
    summary_df.at[idx, "same_sign_positive_folds_vs_m0"] = int(sum(fold_auc_lookup[(model_key, f)] - fold_auc_lookup[("m0_random_forest_frozen", f)] > 0 for f in range(5)))
    summary_df.at[idx, "same_sign_positive_folds_vs_m1"] = int(sum(fold_auc_lookup[(model_key, f)] - fold_auc_lookup[("m1_logistic_regression_baseline", f)] > 0 for f in range(5)))
fold_df = pd.DataFrame(fold_rows)
for idx, row in fold_df.iterrows():
    fold = int(row["fold"])
    model_key = row["model_key"]
    fold_df.at[idx, "delta_auc_vs_m0"] = row["roc_auc"] - fold_auc_lookup[("m0_random_forest_frozen", fold)]
    fold_df.at[idx, "delta_auc_vs_m1_baseline"] = row["roc_auc"] - fold_auc_lookup[("m1_logistic_regression_baseline", fold)]

topk_records = []
positive_rate = float(y.mean())
negative_rate = 1 - positive_rate
for model_key, df in all_oofs.items():
    sorted_df = df.sort_values(["y_pred_proba", "Id"], ascending=[False, True]).reset_index(drop=True)
    total_pos = int((sorted_df["y_true"] == 1).sum())
    total_neg = len(sorted_df) - total_pos
    for k in [50, 100, 250, 500]:
        top = sorted_df.head(k)
        pos = int((top["y_true"] == 1).sum())
        topk_records.append({
            "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID, "model_key": model_key,
            "rank_region": "top_k_positive", "cut_label": f"top_{k}",
            "n_selected": k, "positives_selected": pos, "negatives_selected": k - pos,
            "precision": pos / k, "recall_capture": pos / total_pos,
            "lift_vs_random": (pos / k) / positive_rate,
            "random_precision_baseline": positive_rate,
        })
    for pct in [0.1, 0.2]:
        k = int(round(len(sorted_df) * pct))
        top = sorted_df.head(k)
        pos = int((top["y_true"] == 1).sum())
        topk_records.append({
            "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID, "model_key": model_key,
            "rank_region": "top_quantile_positive", "cut_label": f"top_{int(pct*100)}pct",
            "n_selected": k, "positives_selected": pos, "negatives_selected": k - pos,
            "precision": pos / k, "recall_capture": pos / total_pos,
            "lift_vs_random": (pos / k) / positive_rate,
            "random_precision_baseline": positive_rate,
        })
topk_df = pd.DataFrame(topk_records)

slice_records = []
for public_name, col, min_n in slice_specs:
    for value in sorted(diag[col].dropna().unique(), key=lambda x: str(x)):
        ids = set(diag.loc[diag[col] == value, "Id"].astype(int))
        m0_auc, n, n_pos, n_neg, status = compute_slice_auc(all_oofs["m0_random_forest_frozen"], ids)
        m1_auc, _, _, _, _ = compute_slice_auc(all_oofs["m1_logistic_regression_baseline"], ids)
        for model_key, df in all_oofs.items():
            auc, _, _, _, model_status = compute_slice_auc(df, ids)
            delta_m0 = auc - m0_auc if status == "computed" and model_status == "computed" else np.nan
            delta_m1 = auc - m1_auc if status == "computed" and model_status == "computed" else np.nan
            slice_records.append({
                "experiment_id": EXPERIMENT_ID,
                "run_id": RUN_ID,
                "model_key": model_key,
                "slice_name": public_name,
                "slice_value": str(value),
                "n": n,
                "n_positive": n_pos,
                "n_negative": n_neg,
                "positive_rate": n_pos / n if n else np.nan,
                "roc_auc": auc,
                "delta_auc_vs_m0": delta_m0,
                "delta_auc_vs_m1_baseline": delta_m1,
                "status": model_status,
                "drop_gt_0_02_vs_m0": bool(model_status == "computed" and delta_m0 < -0.02),
                "fragile_positive_support_lt_20": bool(model_status == "computed" and n_pos < 20),
                "notes": "School diagnostic only, not feature" if public_name == "frequent_vs_rare_school_group" else "",
            })
slice_df = pd.DataFrame(slice_records)

dist_records = []
for model_key, df in all_oofs.items():
    probs = df["y_pred_proba"].astype(float)
    q = probs.quantile([0, .01, .05, .1, .25, .5, .75, .9, .95, .99, 1]).to_dict()
    dist_records.append({
        "experiment_id": EXPERIMENT_ID, "run_id": RUN_ID, "model_key": model_key,
        "n": len(df), "mean_pred": float(probs.mean()), "std_pred": float(probs.std(ddof=0)),
        "min": q[0], "p01": q[.01], "p05": q[.05], "p10": q[.1], "p25": q[.25],
        "p50": q[.5], "p75": q[.75], "p90": q[.9], "p95": q[.95], "p99": q[.99],
        "max": q[1], "brier_score": float(brier_score_loss(df["y_true"], probs)),
    })
dist_df = pd.DataFrame(dist_records)

hpo_df = pd.concat([
    m1_hpo_df.assign(run_id=RUN_ID, experiment_id=EXPERIMENT_ID, budget_mode="Standard"),
    cat_hpo_df.assign(run_id=RUN_ID, experiment_id=EXPERIMENT_ID, budget_mode="Standard"),
], ignore_index=True, sort=False)

m1_baseline_auc = float(summary_df.loc[summary_df["model_key"] == "m1_logistic_regression_baseline", "oof_auc"].iloc[0])
m0_auc = float(summary_df.loc[summary_df["model_key"] == "m0_random_forest_frozen", "oof_auc"].iloc[0])
cat_baseline_auc = float(summary_df.loc[summary_df["model_key"] == "catboost_baseline", "oof_auc"].iloc[0])
m1_tuned_auc = float(summary_df.loc[summary_df["model_key"] == "m1_logistic_regression_tuned", "oof_auc"].iloc[0])
cat_tuned_auc = float(summary_df.loc[summary_df["model_key"] == "catboost_tuned", "oof_auc"].iloc[0])

m1_repeated_cv_rows = []
for repeat_seed in [7, 2025, 90210]:
    repeat_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=repeat_seed)
    repeat_oof = np.full(len(X), np.nan, dtype=float)
    repeat_fold_scores = []
    for repeat_fold, (train_idx, valid_idx) in enumerate(repeat_splitter.split(X, y)):
        lr = LogisticRegression(
            C=float(m1_best_config["C"]),
            penalty=str(m1_best_config["penalty"]),
            class_weight=m1_best_config["class_weight"],
            solver=str(m1_best_config["solver"]),
            max_iter=int(m1_best_config["max_iter"]),
            random_state=PROJECT_SEED,
            n_jobs=None,
        )
        pipe = Pipeline([
            ("preprocess", preprocessor_m1),
            ("model", lr),
        ])
        pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = positive_class_proba(pipe.named_steps["model"], pipe.named_steps["preprocess"].transform(X.iloc[valid_idx]))
        repeat_oof[valid_idx] = proba
        repeat_fold_scores.append(float(roc_auc_score(y.iloc[valid_idx], proba)))
    if np.isnan(repeat_oof).any():
        raise RuntimeError("Repeated-CV confirmation produced missing OOF predictions")
    m1_repeated_cv_rows.append({
        "candidate": "m1_logistic_regression_tuned",
        "splitter_seed": repeat_seed,
        "oof_auc": float(roc_auc_score(y, repeat_oof)),
        "delta_vs_frozen_fold_m1_tuned": float(roc_auc_score(y, repeat_oof) - m1_tuned_auc),
        "fold_mean_auc": float(np.mean(repeat_fold_scores)),
        "fold_std_auc": float(np.std(repeat_fold_scores, ddof=0)),
        "fold_auc_scores": "|".join(f"{v:.12f}" for v in repeat_fold_scores),
    })
m1_repeated_cv_df = pd.DataFrame(m1_repeated_cv_rows)

selection_bias_lines = [
    "# Phase 10 Selection-Bias Warning Report",
    "",
    "Phase 10 used bounded deterministic configuration search because Optuna was not installed in the project `.venv` and modifying `.venv` was prohibited. The search spaces and budgets remained within the signed authorization.",
    "",
    f"- M1 configs evaluated: {len(m1_hpo_df)} / 50 authorized.",
    f"- CatBoost configs evaluated: {len(cat_hpo_df)} / 30 authorized.",
    "- Objective: fixed-fold OOF ROC-AUC only.",
    "- Auxiliary metrics and slices are diagnostic only.",
    "- No leaderboard feedback, no submission, no final winner.",
    "- Search-space widening after results: not performed.",
    "- Multiplicity warning: HPO reuses the same frozen CV substrate; independent recomputation and Opus strategic review are required before acceptance.",
    "",
    "## Secondary Repeated-CV Confirmation For Tuned M1",
    "",
    "These splitter seeds are diagnostic only and do not replace the frozen-fold decision substrate.",
    "",
    markdown_table(m1_repeated_cv_df.to_dict("records"), ["candidate", "splitter_seed", "oof_auc", "delta_vs_frozen_fold_m1_tuned", "fold_mean_auc", "fold_std_auc"]),
]
safe_write_text("\n".join(selection_bias_lines) + "\n", paths["selection_bias_report"])

candidate_log_rows = []
for model_key, base_key, family, trials, strategy in [
    ("m1_logistic_regression_tuned", "m1_logistic_regression_baseline", "LogisticRegression", len(m1_hpo_df), "bounded_deterministic_config_search"),
    ("catboost_tuned", "catboost_baseline", "CatBoost", len(cat_hpo_df), "limited_deterministic_config_search_separate_env"),
]:
    row = summary_df.loc[summary_df["model_key"] == model_key].iloc[0]
    base_auc = float(summary_df.loc[summary_df["model_key"] == base_key, "oof_auc"].iloc[0])
    candidate_log_rows.append({
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "candidate_family": family,
        "variant_id": model_key,
        "base_candidate": base_key,
        "features_used": "F2",
        "School_used_as_feature": False,
        "external_data_used": False,
        "leaderboard_used": False,
        "hpo_strategy": strategy,
        "hpo_budget_mode": "Standard",
        "hpo_trials_or_configs": trials,
        "cv_protocol": "frozen_5fold_sha16_96937649526bcadb",
        "fold_sha": FOLD_SHA16_EXPECTED,
        "primary_metric": "ROC-AUC positive-class probability Drafted=1",
        "oof_auc": row["oof_auc"],
        "delta_vs_m0": row["delta_vs_m0"],
        "delta_vs_m1": row["delta_vs_m1"],
        "delta_vs_current_candidate_baseline_if_applicable": row["oof_auc"] - base_auc,
        "fold_mean_auc": row["fold_mean_auc"],
        "fold_std_auc": row["fold_std_auc"],
        "same_sign_positive_folds_vs_m1": row["same_sign_positive_folds_vs_m1"],
        "slice_guard_status": "triggered" if bool(slice_df[(slice_df["model_key"] == model_key) & (slice_df["drop_gt_0_02_vs_m0"])].shape[0]) else "clear",
        "leakage_checks_passed": True,
        "selection_bias_warning_status": "warning_hpo_same_cv_substrate",
        "artifacts_manifest_path": str(paths["artifact_manifest"].relative_to(REPO_ROOT)),
        "validation_report_path": str(paths["validation_report"].relative_to(REPO_ROOT)),
        "notes": "No final winner selected; awaiting independent recomputation and Opus review.",
    })
candidate_log_df = pd.DataFrame(candidate_log_rows)

validation_lines = [
    "# Phase 10 Model Optimization Validation Report",
    "",
    "## Executive Summary",
    "",
    "Phase 10 executed the authorized Standard-budget bounded HPO. No final winner was selected and no submission was authorized.",
    "",
    f"- HEAD: `{head}`",
    f"- Run ID: `{RUN_ID}`",
    f"- M1 baseline AUC: `{m1_baseline_auc:.12f}`",
    f"- M1 tuned AUC: `{m1_tuned_auc:.12f}`",
    f"- CatBoost baseline AUC: `{cat_baseline_auc:.12f}`",
    f"- CatBoost tuned AUC: `{cat_tuned_auc:.12f}`",
    f"- M0 anchor AUC: `{m0_auc:.12f}`",
    "",
    "## Model Summary",
    "",
    markdown_table(summary_df.to_dict("records"), ["model_key", "run_status", "oof_auc", "delta_vs_m0", "delta_vs_m1", "fold_mean_auc", "fold_std_auc", "same_sign_positive_folds_vs_m0", "same_sign_positive_folds_vs_m1"]),
    "",
    "## Secondary Repeated-CV Confirmation",
    "",
    "Repeated-CV confirmation was run for tuned M1 only as a selection-bias diagnostic; it is not a replacement for frozen-fold OOF scoring.",
    "",
    markdown_table(m1_repeated_cv_df.to_dict("records"), ["candidate", "splitter_seed", "oof_auc", "delta_vs_frozen_fold_m1_tuned", "fold_mean_auc", "fold_std_auc"]),
    "",
    "## B5 CatBoost Diagnosis",
    "",
    catboost_hpo_reason,
    "",
    markdown_table(catboost_robust_warnings[["slice_name", "slice_value", "n", "n_positive", "catboost_auc", "catboost_delta_vs_m0"]].to_dict("records"), ["slice_name", "slice_value", "n", "n_positive", "catboost_auc", "catboost_delta_vs_m0"], max_rows=20),
    "",
    "## Scope And Leakage",
    "",
    "- F2 only.",
    "- School excluded from all feature matrices.",
    "- All learned preprocessing fitted inside each training fold.",
    "- No external data, no leaderboard, no submissions, no Phase 11.",
    "- `logs/experiment_log.csv` was read before execution and verified unchanged after artifact writing.",
    "",
    "Phase 10 execution complete. No final winner selected, no submission authorized. Candidates remain phase-gated. Awaiting independent recomputation and Opus strategic review. Phase 11 remains locked.",
]
safe_write_text("\n".join(validation_lines) + "\n", paths["validation_report"])

safe_write_csv(m1_best_oof, paths["m1_oof"])
safe_write_csv(cat_best_oof, paths["catboost_oof"])
safe_write_csv(hpo_df, paths["hpo_results"])
safe_write_csv(summary_df, paths["model_summary"])
safe_write_csv(fold_df, paths["fold_metrics"])
safe_write_csv(slice_df, paths["slice_report"])
safe_write_csv(topk_df, paths["topk_quantile"])
safe_write_csv(dist_df, paths["score_distribution"])
safe_write_csv(candidate_log_df, paths["candidate_log"])

manifest_records = []
for key, path in paths.items():
    if key in {"notebook", "artifact_manifest"}:
        continue
    manifest_records.append({
        "experiment_id": EXPERIMENT_ID,
        "run_id": RUN_ID,
        "artifact_key": key,
        "path": str(path.relative_to(REPO_ROOT)).replace("\\", "/"),
        "sha256": sha256_file(path),
        "row_count": csv_rows(path) if path.suffix.lower() == ".csv" else "",
        "created_utc": datetime.now(timezone.utc).isoformat(),
    })
manifest_records.append({
    "experiment_id": EXPERIMENT_ID,
    "run_id": RUN_ID,
    "artifact_key": "artifact_manifest",
    "path": str(paths["artifact_manifest"].relative_to(REPO_ROOT)).replace("\\", "/"),
    "sha256": "self_excluded_until_after_write",
    "row_count": "",
    "created_utc": datetime.now(timezone.utc).isoformat(),
})
safe_write_csv(pd.DataFrame(manifest_records), paths["artifact_manifest"])

if sha256_file(main_log_path) != main_log_sha_before:
    raise RuntimeError("logs/experiment_log.csv changed during Phase 10")
post_forbidden = run_cmd(["git", "diff", "--name-only", "--", "data/input", "notebooks/_official", "references", "outputs/submissions", "logs/experiment_log.csv", ".vscode/settings.json"])
if post_forbidden:
    raise RuntimeError(f"Forbidden-path diff after execution: {post_forbidden}")

print("Phase 10 artifacts written")
print(summary_df[["model_key", "oof_auc", "delta_vs_m0", "delta_vs_m1"]].to_string(index=False))

C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ra

C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ra

C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ra

...

C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ra

C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\GitHub\reto_Tokio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ra

Phase 10 artifacts written
                      model_key  oof_auc  delta_vs_m0  delta_vs_m1
        m0_random_forest_frozen 0.811650     0.000000    -0.015432
m1_logistic_regression_baseline 0.827082     0.015432     0.000000
              catboost_baseline 0.820294     0.008644    -0.006788
   m1_logistic_regression_tuned 0.827482     0.015832     0.000400
                 catboost_tuned 0.830321     0.018671     0.003239


### Interpretation

- **Main result:** Phase 10 artifacts are written under the authorized namespace.
- **Methodological reading:** Tuned candidates remain phase-gated evidence, not final winners.
- **Risk or warning:** HPO uses the same frozen CV substrate, so independent recomputation and Opus review are required before acceptance.
- **Diagnostic decision:** Stop after verification; no staging, commit, submission, or Phase 11 action.